In [5]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# Tavily gives the agent a web search capability.
# The agent can use this tool to answer live questions with external information.
search = TavilySearchResults(max_results=3)
search.invoke("Who are the top stars of the 2024 Eurocup?")


[{'url': 'https://www.espn.com/soccer/story/_/id/40319301/euro-2024-power-rankings-france-top-germany-england',
  'content': "## 1. FRANCE\n\nBut for a handful of adjustments, France head into Euro 2024 looking like the same force that came within a penalty kick of winning the 2022 World Cup. They're still tactically disciplined, physical, stacked with quality and boasting one of the world's best forwards in Kylian Mbappé. With that in mind, they're still favourites in our eyes.\n\nDidier Deschamps' men will need to shake off the disappointment of a quarterfinal exit at Euro 2020, but you get the impression that, with Olivier Giroud back in and Karim Benzema back out, this team will hum a more familiar tune. That tune -- a cautious one that doesn't always excite -- isn't universally popular, but it has proven to be devastatingly effective. [...] Some would argue they're caught between two generations; others could say they boast a nice blend of youth and experience. It's up to manager 

In [ ]:
# Attach the available tools to the agent.
# The model can decide whether to call the tool or answer directly.
tools = [search]


In [8]:
tools

[TavilySearchResults(max_results=3)]

In [ ]:
from langgraph.prebuilt import create_react_agent

# ReAct agent = reasoning + action loop.
# The LLM can reason, decide to call a tool, and then continue with the result.
agent_executor = create_react_agent(llm, tools)


In [ ]:
from langchain_core.messages import HumanMessage

# Ask a factual question that may require live search.
response = agent_executor.invoke({"messages": [HumanMessage(content="Where is the soccer Eurocup 2024 played?")]})

response["messages"]

[HumanMessage(content='Where is the soccer Eurocup 2024 played?', id='65ce1a8c-7709-4c30-ac89-705687175745'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_NbRJXViwu4LOf624WvQfc7XZ', 'function': {'arguments': '{"query":"Eurocup 2024 location"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 90, 'total_tokens': 113, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6d10964534', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-79610c60-bda3-4238-8acd-fc523d1617a9-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Eurocup 2024 location'}, 'id': 'call_NbRJXViwu4LOf624WvQfc7XZ', 'type': 'tool_call'}], usage_metadata={'input_t

In [ ]:
from langchain_core.messages import HumanMessage

# This is a general knowledge question, so the agent may answer without calling the tool.
response = agent_executor.invoke({"messages": [HumanMessage(content="What is Python?")]})

response["messages"]

[HumanMessage(content='What is Python?', id='f57a5646-a887-4e6f-b400-76e9bc6a254a'),
 AIMessage(content="Python is a high-level, interpreted programming language known for its readability, simplicity, and versatility. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming. It is widely used for web development, data analysis, artificial intelligence, scientific computing, automation, and more. Python's extensive standard library and active community contribute to its popularity and make it suitable for both beginners and experienced developers.", response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 83, 'total_tokens': 181, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gp

In [ ]:
# Streaming shows the agent's step-by-step reasoning and tool use in real time.
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="When and where will it be the 2024 Eurocup final match?")]}
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_exNpvY8zMMUTyCd4XQVNM2Dj', 'function': {'arguments': '{"query":"2024 Eurocup final match date and location"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 94, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6d10964534', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-8ec689b7-4f4a-4847-b85f-383af01406f8-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': '2024 Eurocup final match date and location'}, 'id': 'call_exNpvY8zMMUTyCd4XQVNM2Dj', 'type': 'tool_call'}], usage_metadata={'input_tokens': 94, 'output_tokens': 26, 'total_token

## Adding memory
* Adding memory in LangGraph is very similar to what we did with LangChain.
* Memory lets the agent keep context across multiple turns in the same conversation.
* This is useful when the user asks follow-up questions that depend on earlier messages.


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# MemorySaver stores conversation state so the agent can remember prior messages.
memory = MemorySaver()

In [ ]:
# Recreate the agent with a checkpointer so it remembers the conversation state.
agent_executor = create_react_agent(llm, tools, checkpointer=memory)

# Each thread_id represents a separate chat memory session.
config = {"configurable": {"thread_id": "001"}}


In [ ]:
# Ask the agent a question in thread 001 and keep the history for this conversation.
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="Who won the 2024 soccer Eurocup?")]}, config
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='I am sorry for the inconvenience. Let me find the information about the winner of the 2024 soccer Eurocup for you.', additional_kwargs={'tool_calls': [{'id': 'call_gb9W8DQ0zx0orw1nRwXDRYJy', 'function': {'arguments': '{"query":"2024 soccer Eurocup winner"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 929, 'total_tokens': 979, 'completion_tokens_details': {'audio_tokens': 0, 'reasoning_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-ce696981-8dd4-4319-b62a-b0f2be73caa9-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': '2024 soccer Eurocup winner'}, 'id': 'call_gb9W8DQ0zx0orw1nRwXDRYJy', 'type': 'tool_call'}], us

In [ ]:
# This follow-up question depends on the previous answer, so memory helps the agent stay in context.
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="Who were the top stars of that winner team?")]}, config
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_bI7uTHk2LrxI9B4kpdBIrBHh', 'function': {'arguments': '{"query":"top players of Spain national team Euro 2024"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 1764, 'total_tokens': 1791, 'completion_tokens_details': {'audio_tokens': 0, 'reasoning_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-0efd7059-7feb-42aa-a933-993e716ee6cf-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'top players of Spain national team Euro 2024'}, 'id': 'call_bI7uTHk2LrxI9B4kpdBIrBHh', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1764, 'output_tokens': 27, 'total_tokens': 179

In [ ]:
# Different thread_id means a different conversation context.
config = {"configurable": {"thread_id": "002"}}
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="About what soccer team we were talking?")]}, config
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='We were discussing the Spain national soccer team and their victory in the 2024 soccer Eurocup. If you have any more questions or need further information, feel free to ask!', response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 2141, 'total_tokens': 2178, 'completion_tokens_details': {'audio_tokens': 0, 'reasoning_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-03a75b3f-3017-4a6a-a39a-84f1060ffd0b-0', usage_metadata={'input_tokens': 2141, 'output_tokens': 37, 'total_tokens': 2178})]}}
----
